In [1]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, read_in_routine, get_metadata
from plot_icusleep_data import icu_sleep_plot
from scipy.stats import variation
from plotly.offline import plot
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
import re

%load_ext autoreload
%autoreload 2

In [2]:
def sleep_indices(stages, verbose=False):

    stages = stages.dropna()

    hours_data_available = stages.shape[0]/fs/3600
    hours_sleep = sum(stages<5)/fs/3600

    if hours_sleep == 0:
        return [hours_data_available, hours_sleep, 
               np.nan, np.nan, np.nan, np.nan, 
               np.nan, np.nan, np.nan]

    perc_n1 = np.round((sum(stages==3)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_n2 = np.round((sum(stages==2)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_n3 = np.round((sum(stages==1)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_r  = np.round((sum(stages==4)/fs/3600) / hours_sleep * 100, 0).astype('int64')

    # sleep fragmentation index:
    w_or_n1 = np.isin(stages,[5,3])
    deep_sleep = np.isin(stages,[1,2,4])
    fragmentation_shift = np.logical_and(w_or_n1[1:], deep_sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos)
    sfi = np.round(len(fragmentation_pos)/hours_sleep, 1)

    # SFI2, based only on transitions to W from N2, N3 or R
    stages_without_n1 = stages[np.isin(stages,[1,2,4,5])]
    w = np.isin(stages_without_n1,[5])
    deep_sleep = np.isin(stages_without_n1,[1,2,4])
    fragmentation_shift = np.logical_and(w[1:], deep_sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos, fs=10)
    sfi2 = np.round(len(fragmentation_pos)/hours_sleep, 1)
    
    # arousal index, based  on transitions to W from sleep
    w = np.isin(stages,[5])
    sleep = np.isin(stages,[1,2,3,4])
    fragmentation_shift = np.logical_and(w[1:], sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos, fs=10)
    arousal_index = np.round(len(fragmentation_pos)/hours_sleep, 1)

    hours_sleep = np.round(hours_sleep, 10)
    hours_data_available = np.round(hours_data_available, 1)

    if verbose: 
        print(f'Hours of data available:     {hours_data_available:.1f}')
        print(f'Hours of sleep:              {hours_sleep:.10f}')
        print(f'N1, N2, N3, R in %:          {[perc_n1, perc_n2, perc_n3, perc_r]}')
        print(f'Sleep Fragmentation Index:   {sfi}')
        print(f'Sleep Fragmentation Index2:  {sfi2}')
        print(f'Arousal Index:               {arousal_index}')

    return [hours_data_available, hours_sleep, 
           perc_n1, perc_n2, perc_n3, perc_r, 
           sfi, sfi2, arousal_index]

In [3]:
cohort = 'icu'
data_dir = f'/media/ssd2/{cohort}_final_files'
data_dir = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step_5_ecg_sleepstaging_nohrv/'

# plots_dir = f'/media/mad3/{cohort}_plots_sleep_stages'
plots_dir = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/sleep_staging/plots_08_23'

# cohort = 'covid_resp'
# # data_dir = '/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data_2'
# data_dir = '/media/ssd2/Combined_Data5'

# plots_dir = f'/media/mad3/Projects/covid_data/{cohort}_plots_sleep_stages_comb5'
# # plots_dir = '/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data_2_Plots'

if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)

files = os.listdir(data_dir)
files.sort()
len(files)

12

In [4]:
i = 0

In [5]:
summary = pd.DataFrame(columns = ['study_id', 'pred_var', 'hours_data_available', 'hours_sleep', 
           'perc_n1', 'perc_n2', 'perc_n3', 'perc_r', 
           'sfi', 'sfi2', 'arousal_index'])

prediction_vars = ['stage_pred_amendsumeffort', 'stage_pred_ecg_nn', 'stage_pred_activity10sec']

In [6]:
for file in files:

    study_id = re.search('\d\d\d', file)[0]
    data, hdr, fs = read_in_routine(os.path.join(data_dir, file))

    data = data.loc[pd.notna(data.stage_pred_ecg_nn) & pd.notna(data.stage_pred_amendsumeffort)]

    for prediction_var in prediction_vars:

        i += 1
        [hours_data_available, hours_sleep, 
               perc_n1, perc_n2, perc_n3, perc_r, 
               sfi, sfi2, arousal_index] = sleep_indices(data[prediction_var])

        summary.loc[i] = [study_id, prediction_var.replace('stage_pred_', ''), hours_data_available, hours_sleep, 
               perc_n1, perc_n2, perc_n3, perc_r, 
               sfi, sfi2, arousal_index]

In [16]:
summary = summary[summary.hours_data_available > 0]

In [13]:
if 0:
    summary.to_csv('summary_staging_comparison_ecg_resp.csv', index=False)

In [24]:
def sleep_summary_barplots(ax, sel, variables, x_labels,  color='navy'):
    
    num = len(variables)
    ax.bar(np.arange(num), sel[variables].mean(axis=0), yerr=[np.zeros(num,), sel[variables].std(axis=0)], color=color)
    ax.set_xticks(np.arange(num))
    ax.set_xticklabels(x_labels, rotation=90)
#     ax.set_title(f'ICU (n={len(pd.unique(sel.study_id))})')

    for iV in range(num):
        var_mean = np.int(np.round(sel[variables[iV]].mean()))
        if var_mean > 15000: # 5:
            ax.annotate(str(var_mean), xy=(iV, var_mean-10), color='white', ha='center')
        else:
            ax.annotate(str(var_mean), xy=(iV, np.min([var_mean+25, 98])), color='black', ha='center')
    return ax


In [27]:

# stage_v = 'SSComb1'

# variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
#        f'N2{stage_v}', f'N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']

variables = ['hours_data_available', 'hours_sleep', 
           'perc_n1', 'perc_n2', 'perc_n3', 'perc_r', 
           'sfi', 'arousal_index']

x_labels = ['Data (h)', 'Sleep (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']

fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(9,3), sharey=True)

sel = summary.loc[summary.pred_var == 'amendsumeffort' ]
ax1 = sleep_summary_barplots(ax1, sel, variables, x_labels,  color='navy')
ax1.set_title(f'Breathing (n={len(pd.unique(sel.study_id))})')
sel = summary.loc[summary.pred_var == 'ecg_nn' ]
ax2 = sleep_summary_barplots(ax2, sel, variables, x_labels,  color='red')
ax2.set_title(f'HRV (n={len(pd.unique(sel.study_id))})')
sel = summary.loc[summary.pred_var == 'activity10sec' ]
ax3 = sleep_summary_barplots(ax3, sel, variables, x_labels,  color='gray')
ax3.set_title(f'Actigraphy (n={len(pd.unique(sel.study_id))})')

plt.subplots_adjust(bottom=0.27)
plt.tight_layout()

# plt.savefig('sleep_stage_distribution.png', dpi=300)

# fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(12,3), sharey=True)

# stage_v = 'SSComb1'
# variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
#        f'N2{stage_v}', f'N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
# x_labels = ['Sleep (h)', 'N1 (%)',
#        'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']

# sel = Xy_icu
# ax1 = sleep_summary_barplots(ax1, sel, variables, x_labels,  color='navy')
# ax1.set_title(f'ICU (n={len(pd.unique(sel.study_id))})')
# sel = Xy_sleeplab
# ax2 = sleep_summary_barplots(ax2, sel, variables, x_labels,  color='goldenrod')
# ax2.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))}), Model')

# stage_v = 'SSExpert'
# variables = [f'h_sleep_{stage_v}', f'N1{stage_v}',
#        f'N2{stage_v}', f'N3{stage_v}', f'R{stage_v}', f'SFI{stage_v}', f'ArousalIndex{stage_v}']
# x_labels = ['Sleep (h)', 'N1 (%)',
#        'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']
# ax3 = sleep_summary_barplots(ax3, sel, variables, x_labels,  color='forestgreen')
# ax3.set_title(f'Sleeplab (n={len(pd.unique(sel.study_id))}), Expert')

# plt.subplots_adjust(bottom=0.27)
# plt.tight_layout()

plt.savefig('sleep_stage_comparison_summary.png', dpi=300)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [28]:
variables = ['hours_data_available', 'hours_sleep', 
           'perc_n1', 'perc_n2', 'perc_n3', 'perc_r', 
           'sfi', 'arousal_index']

x_labels = ['Data (h)', 'Sleep (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal I.']


study_ids = summary.study_id.unique()


fig, ax = plt.subplots(len(study_ids), 3, figsize=(9,2*len(study_ids)), sharey=True)

for iS, study_id in enumerate(study_ids):
    sel = summary.loc[(summary.study_id == study_id) & (summary.pred_var == 'amendsumeffort')]
    ax[iS, 0] = sleep_summary_barplots(ax[iS, 0], sel, variables, x_labels,  color='navy')
    ax[iS, 0].set_title(study_id)
    sel = summary.loc[(summary.study_id == study_id) & (summary.pred_var == 'ecg_nn')]
    ax[iS, 1] = sleep_summary_barplots(ax[iS, 1], sel, variables, x_labels,  color='red')
#     ax[iS, 1].set_title(f'HRV (n={len(pd.unique(sel.study_id))})')
    sel = summary.loc[(summary.study_id == study_id) & (summary.pred_var == 'activity10sec')]
    ax[iS, 2] = sleep_summary_barplots(ax[iS, 2], sel, variables, x_labels,  color='gray')
#     ax[iS, 2].set_title(f'Actigraphy (n={len(pd.unique(sel.study_id))})')
# 
plt.suptitle('Sleep Staging Comparison: Breathing, HRV, Actigraphy')
plt.subplots_adjust(bottom=0.27)
plt.tight_layout()

plt.savefig('sleep_stage_comparison_single_subjects.png', dpi=300)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …